# 第 48 课｜推理服务接口、请求生命周期与端到端预算

**从“一个脚本能运行”，到“多个调用方可以有序地请求它”。**

本课接续第 47 课的评测，增加真实的本机 HTTP 服务，研究提交、排队、截止时间、取消与资源预留。
默认使用标准库演示后端；内容是固定文本，不是模型回答。真实 Qwen 后端是可选路径，不会默认下载。

**交付**：本 Notebook 与同名 `.py` 都可独立使用。本讲义内置全部实现，不导入旁边的脚本。

**本次验证边界**：默认流程在真实回环 HTTP 上执行；不等于真实模型、GPU、生产负载或网络故障全覆盖。

## 0. 学习目标与运行习惯

本课结束时，你应能解释：

1. HTTP `202`、请求状态 `succeeded`、任务语义成功为什么不同。
2. 为什么一个推理 worker 仍然需要处理多条控制连接。
3. 排队为什么也要消耗同一份截止预算；客户端超时为什么不是服务端取消。
4. 为什么运行中收到停止信号以后，仍不能立即释放执行槽。
5. 同一幂等键如何避免确认丢失后的重复推理，以及它在重启后的边界。

先完成默认模式，不开启真实模型。所有实验服务用 `with` 自动关闭，不会在 Notebook 中留下无限运行的服务器。
如果当前环境禁止创建本机 socket，HTTP 单元会明确失败；本课不会用假响应替代成功结果。

本课只监听 `127.0.0.1`。HTTP 标准库服务仅供教学，不应用来对公网提供生产服务。
[Python 官方说明](https://docs.python.org/3.13/library/http.server.html)

## 1. 请求不是函数返回值：先确定接口语义

同步调用 `answer = generate(prompt)` 会等待计算返回。本课把“接收请求”和“查看结果”拆开：

| 接口 | 用途 |
|---|---|
| `GET /healthz` | HTTP 进程能否响应，不等于模型能成功回答 |
| `GET /readyz` | 后端已加载、worker 存活、服务允许接收 |
| `POST /v1/requests` | 校验、预留并排队，返回请求 ID |
| `GET /v1/requests/{id}` | 查看状态与已保存结果 |
| `POST /v1/requests/{id}/cancel` | 请求停止，不是强杀线程 |
| `GET /v1/metrics` | 查看当前队列、预留、计数 |

`202 Accepted` 只表示已经接受处理，不保证最终成功。`503` 在本课用于过载等服务不可用条件；
临时排队拥堵附上 `Retry-After` 建议，不表示客户端一定要重试。
这些语义可对照 [RFC 9110](https://www.rfc-editor.org/rfc/rfc9110.html)。

本课是自己的异步作业 API，**不是 OpenAI 兼容接口**，也不是可以多进程扩容的生产服务。

In [1]:
import argparse
import copy
import hashlib
import hmac
import http.client
import json
import math
import os
import platform
import re
import secrets
import socket
import sys
import threading
import time
import uuid
from collections import Counter, deque
from contextlib import contextmanager
from dataclasses import dataclass, field
from datetime import datetime, timezone
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from typing import Any, Callable
from urllib.parse import urlsplit

FORMAT = "lesson48-service-v1"
MODEL_ID = "Qwen/Qwen3-0.6B"
REVISION = "c1899de289a04d12100db370d81485cdf75e47ca"
TERMINAL = frozenset({"succeeded", "cancelled", "expired", "failed"})
STATES = frozenset({"queued", "running", "stopping"}) | TERMINAL
SOURCES = {
    "http": "https://www.rfc-editor.org/rfc/rfc9110.html",
    "http_server": "https://docs.python.org/3.13/library/http.server.html",
    "time": "https://docs.python.org/3.13/library/time.html#time.monotonic",
    "threads": "https://docs.python.org/3.13/library/concurrent.futures.html",
    "generation": "https://huggingface.co/docs/transformers/v4.57.3/en/main_classes/text_generation",
    "stopping": "https://huggingface.co/docs/transformers/v4.57.3/en/internal/generation_utils",
    "qwen": f"https://huggingface.co/{MODEL_ID}/blob/{REVISION}/README.md",
    "cuda": "https://docs.pytorch.org/docs/stable/notes/cuda.html",
}


class ServiceError(ValueError):
    def __init__(self, status: int, code: str, message: str, retryable: bool = False):
        super().__init__(message)
        self.status, self.code, self.retryable = status, code, retryable


def require(ok: bool, code: str, message: str, status: int = 400) -> None:
    if not ok:
        raise ServiceError(status, code, message)


def canonical(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)


def digest(value: Any) -> str:
    return hashlib.sha256(canonical(value).encode("utf-8")).hexdigest()


def strict_json(text: str) -> Any:
    def pairs(items):
        result = {}
        for key, value in items:
            require(key not in result, "duplicate_key", "JSON 不允许重复键")
            result[key] = value
        return result
    def bad(_):
        raise ServiceError(400, "nonfinite", "JSON 不允许非有限值")
    def finite(text):
        n = float(text)
        if not math.isfinite(n):
            bad(text)
        return n
    try:
        return json.loads(text, object_pairs_hook=pairs, parse_constant=bad,
                          parse_float=finite)
    except (json.JSONDecodeError, UnicodeError, RecursionError, OverflowError) as exc:
        raise ServiceError(400, "invalid_json", "需要完整合法的 UTF-8 JSON") from exc


def validate_request(body: Any) -> dict:
    require(type(body) is dict and set(body) == {"prompt", "max_new_tokens", "timeout_ms"},
            "schema", "字段必须恰好为 prompt、max_new_tokens、timeout_ms")
    prompt, n, ms = body["prompt"], body["max_new_tokens"], body["timeout_ms"]
    require(type(prompt) is str and bool(prompt.strip()), "prompt", "prompt 需要非空文本")
    try:
        size = len(prompt.encode("utf-8"))
    except UnicodeError as exc:
        raise ServiceError(400, "prompt", "prompt 含非法 Unicode") from exc
    require(size <= 4096, "prompt_too_large", "prompt UTF-8 上限 4096 字节", 413)
    require(type(n) is int and 1 <= n <= 256, "output_limit", "新增上限为 1–256 的整数")
    require(type(ms) is int and 25 <= ms <= 120000, "timeout", "timeout_ms 必须为 25–120000 的整数")
    return dict(body)


class WorkBudget:
    """单次 Agent 工作流预算示例：所有子步骤共享同一单调时钟截止时间。
    units 是每次预留的 输入+最大输出 之和，不是实际账单或显存估算。
    本类供串行控制器使用；HTTP 服务不执行任意工具。
    """
    def __init__(self, timeout_ms: int, max_model_calls: int, max_tool_calls: int,
                 max_units: int, clock: Callable[[], float] = time.monotonic):
        for value in (timeout_ms, max_model_calls, max_tool_calls, max_units):
            require(type(value) is int and value >= 0, "budget", "预算必须是非负整数")
        require(timeout_ms > 0, "budget", "需要正的截止预算")
        self.clock = clock
        self.deadline = clock() + timeout_ms / 1000
        self.model_limit, self.tool_limit, self.unit_limit = max_model_calls, max_tool_calls, max_units
        self.model_calls = self.tool_calls = self.reserved_units = 0

    def remaining_ms(self) -> int:
        return max(0, math.floor((self.deadline - self.clock()) * 1000 + 1e-7))

    def claim(self, kind: str, input_units: int = 0, max_new_units: int = 0) -> dict:
        require(kind in {"model", "tool"}, "kind", "未知子步骤")
        for value in (input_units, max_new_units):
            require(type(value) is int and value >= 0, "units", "单位数需为非负整数")
        remaining = self.remaining_ms()
        require(remaining > 0, "deadline", "整个工作流已过期", 408)
        if kind == "model":
            require(self.model_calls < self.model_limit, "model_budget", "模型调用预算耗尽", 409)
            cost = input_units + max_new_units
            require(self.reserved_units + cost <= self.unit_limit,
                    "unit_budget", "累计预留单位数超限", 409)
            self.model_calls += 1
            self.reserved_units += cost
        else:
            require(input_units == max_new_units == 0, "units", "工具不在本例预留 token")
            require(self.tool_calls < self.tool_limit, "tool_budget", "工具预算耗尽", 409)
            self.tool_calls += 1
        return {"remaining_ms": remaining, "model_calls": self.model_calls,
                "tool_calls": self.tool_calls, "reserved_units": self.reserved_units}

### 先校验输入，不要让客户端自己决定无限预算

请求恰好包含 `prompt`、`max_new_tokens`、`timeout_ms`；额外字段会被拒绝。
`True` 在 Python 中与整数有继承关系，但不能被当成合法的 token 数。

文本字节数限制用于控制 HTTP/CPU 输入大小；分词后的上下文限制用于控制模型输入。
两者不是同一个单位。真实模型还要把系统消息与 chat template 加进去再计数。

In [2]:
request = {"prompt": "什么是KV cache？", "max_new_tokens": 16, "timeout_ms": 5000}
print(validate_request(request))
for bad in [dict(request, max_new_tokens=True), dict(request, approved=True)]:
    try:
        validate_request(bad)
    except ServiceError as error:
        print(error.status, error.code, str(error))

{'prompt': '什么是KV cache？', 'max_new_tokens': 16, 'timeout_ms': 5000}
400 output_limit 新增上限为 1–256 的整数
400 schema 字段必须恰好为 prompt、max_new_tokens、timeout_ms


## 2. 三种时间限制不要混为一谈

| 限制 | 本课定义 |
|---|---|
| HTTP 连接/读写超时 | 客户端或服务线程等待通信的时间，不保证计算被取消 |
| 请求截止预算 | 从已解析请求进入控制器开始，包含准备、排队和执行 |
| 生成长度上限 | 最多生成多少 token，不保证耗时固定 |

使用单调时钟计算截止预算；墙钟时间只适合人类可读日志，不应用于差值预算。
[`time.monotonic()`](https://docs.python.org/3.13/library/time.html#time.monotonic) 不会因为系统时钟调整而倒退。

下例使用**手动时钟**，用于准确演示预算算术，不是实际性能测量。
在 Agent 中，每个模型调用和工具调用应继承剩余预算，不要每进入一层都重新得到完整的1秒。
跨进程不能直接传另一台机器的 monotonic 绝对值，应协商剩余时限并继续核对本地总截止时间。

In [3]:
# 这里先使用简单列表作为可控时钟，后面的实验封装成 ManualClock。
now = [100.0]
budget = WorkBudget(1000, max_model_calls=2, max_tool_calls=1, max_units=100,
                    clock=lambda: now[0])
print("第一次：", budget.claim("model", input_units=10, max_new_units=20))
now[0] += 0.75
print("750ms后：", budget.claim("tool"))
now[0] += 0.30
try:
    budget.claim("model", input_units=10, max_new_units=20)
except ServiceError as error:
    print("累计1050ms：", error.code)

第一次： {'remaining_ms': 1000, 'model_calls': 1, 'tool_calls': 0, 'reserved_units': 30}
750ms后： {'remaining_ms': 250, 'model_calls': 1, 'tool_calls': 1, 'reserved_units': 30}
累计1050ms： deadline


### Token 预算、等待人数与显存不是同一件事

本课同时限制：单个请求的 `完整输入＋最大新增`、等待队列长度、全部未结束请求的预留单位总数、结果记录总数。

预留数是保守的工作量配额，不是实际生成量，也不是 VRAM 字节。权重、KV cache、激活、工作区、数据精度和分配器都会影响真实显存。
只用一个推理 worker 可以避免本课中多个 `generate()` 同时占用模型，但不能保证单个请求不会 OOM。

HTTP 控制线程与模型 worker 需要区分：前者处理提交/查询/取消，后者才执行模型。本课限制8个控制连接线程，**模型并发固定为1**。

In [4]:
class DemoBackend:
    name = "deterministic-demo"
    unit = "demo_codepoint_unit"

    def __init__(self, step_delay: float = 0.01):
        require(math.isfinite(step_delay) and 0 <= step_delay <= 0.2,
                "delay", "演示延时需在0–0.2秒")
        self.step_delay = step_delay
        self.calls = 0

    def prepare(self, prompt: str) -> dict:
        # 4 个虚拟边界单位。不是 Qwen tokenizer，不能换算模型成本。
        return {"input_units": len(prompt) + 4}

    def generate(self, prepared: dict, max_new: int,
                 stop: Callable[[], str | None], observe: Callable[[int], None]) -> dict:
        self.calls += 1
        chunks = ["演示：", "请求", "已经", "按预算", "完成", "。"]
        out = []
        for chunk in chunks[:max_new]:
            if stop():
                break
            if self.step_delay:
                time.sleep(self.step_delay)  # 一个不可中途取消的演示计算块
            if stop():
                break
            out.append(chunk)
            observe(len(out))
        return {"text": "".join(out), "generated_units": len(out),
                "finish_reason": "demo_end" if len(out) == len(chunks) else "length"}

    def metadata(self) -> dict:
        return {"name": self.name, "unit": self.unit, "step_delay_seconds": self.step_delay,
                "is_language_model": False, "measures_gpu_performance": False}

In [5]:
backend = DemoBackend(step_delay=0)
print("后端身份：", backend.metadata())
print("中文输入的演示单位：", backend.prepare("中文"))
observed = []
result = backend.generate(backend.prepare("中文"), max_new=3,
                          stop=lambda: None, observe=observed.append)
print("结果：", result)
print("累计输出单位：", observed)

后端身份： {'name': 'deterministic-demo', 'unit': 'demo_codepoint_unit', 'step_delay_seconds': 0, 'is_language_model': False, 'measures_gpu_performance': False}
中文输入的演示单位： {'input_units': 6}
结果： {'text': '演示：请求已经', 'generated_units': 3, 'finish_reason': 'length'}
累计输出单位： [1, 2, 3]


## 3. 请求状态机：停止请求与停止完成是两件事

```text
queued ───────→ running ───────→ succeeded / failed
  │                │
  │                └─取消/截止→ stopping ─后端返回→ cancelled / expired
  └─取消/截止────────────────────────────→ cancelled / expired
```

`queued` 没开始后端，可以直接移出队列；`running` 的计算已经开始，只能先发停止信号。
**`stopping` 仍然持有执行槽与预留预算。** 后端真正返回才释放，否则第二个请求可能在第一个仍使用 GPU 时开始。

Python 的 Future 不能直接取消已运行的工作：[官方文档](https://docs.python.org/3.13/library/concurrent.futures.html)。
真实模型的 `StoppingCriteria` 也是协作式检查，不是内核级中断；长 prefill/GPU算子可能超过截止时间后才返回。

不要把暂存的部分文本当作完成答案。终态 `succeeded` 也只表示后端正常完成调用；`finish_reason="length"` 并不保证自然结束。

In [6]:
@dataclass
class Limits:
    max_pending: int = 2
    max_context_units: int = 1024
    max_reserved_units: int = 2048
    max_records: int = 128

    def validate(self):
        for name, value in vars(self).items():
            require(type(value) is int and 1 <= value <= 65536, "limits", f"{name} 无效")
        require(self.max_reserved_units >= self.max_context_units,
                "limits", "总预留预算至少容纳一个最大上下文请求")


@dataclass
class RequestRecord:
    request_id: str
    key: str
    body_hash: str
    prepared: dict
    max_new: int
    received: float
    deadline: float
    reserved: int
    state: str = "queued"
    started: float | None = None
    finished: float | None = None
    first_output: float | None = None
    abort_reason: str | None = None
    generated: int = 0
    result: dict | None = None
    error: dict | None = None
    events: list = field(default_factory=list)


class RequestEngine:
    def __init__(self, backend=None, limits: Limits | None = None,
                 clock: Callable[[], float] = time.monotonic):
        self.backend = backend or DemoBackend()
        self.limits = limits or Limits()
        self.limits.validate()
        self.clock, self.epoch = clock, uuid.uuid4().hex
        self.cv = threading.Condition(threading.RLock())
        self.prepare_lock = threading.Lock()
        self.records: dict[str, RequestRecord] = {}
        self.keys: dict[str, str] = {}
        self.pending = deque()
        self.active: str | None = None
        self.reserved = 0
        self.accepting, self.closed = True, False
        self.counters = Counter()
        self.worker = threading.Thread(target=self._work, name="lesson48-inference", daemon=True)
        self.timer = threading.Thread(target=self._timer, name="lesson48-deadlines", daemon=True)
        self.worker.start()
        self.timer.start()

    def _event(self, rec: RequestRecord, state: str, reason: str | None = None):
        require(state in STATES, "state", "未知状态", 500)
        rec.state = state
        rec.events.append({"state": state, "reason": reason,
                           "elapsed_ms": max(0.0, (self.clock() - rec.received) * 1000)})

    def _finish(self, rec: RequestRecord, state: str, result=None, error=None):
        if rec.state in TERMINAL:
            return
        require(state in TERMINAL, "state", "结束状态错误", 500)
        rec.result, rec.error, rec.finished = result, error, self.clock()
        self._event(rec, state, rec.abort_reason)
        self.reserved -= rec.reserved
        require(self.reserved >= 0, "accounting", "预算重复释放", 500)
        # 不在 HTTP 响应中保存或暴露完整 prompt / token 输入。
        rec.prepared = {"input_units": rec.prepared["input_units"]}
        self.counters["terminal_" + state] += 1
        self.cv.notify_all()

    def _abort(self, rec: RequestRecord, reason: str):
        if rec.state in TERMINAL:
            return
        if rec.abort_reason is None:
            rec.abort_reason = reason
        if rec.state == "queued":
            self._finish(rec, "expired" if rec.abort_reason == "deadline" else "cancelled")
        elif rec.state == "running":
            self._event(rec, "stopping", rec.abort_reason)
        # stopping 继续占 worker 和预留，直到后端实际退出。

    def _sweep(self):
        now = self.clock()
        for rec in self.records.values():
            if rec.state not in TERMINAL and now >= rec.deadline:
                self._abort(rec, "deadline")
        self.pending = deque(rid for rid in self.pending if self.records[rid].state == "queued")

    def _existing(self, key: str, body_hash: str):
        if key not in self.keys:
            return None
        rec = self.records[self.keys[key]]
        require(rec.body_hash == body_hash, "key_conflict", "同一键不能对应不同内容或预算", 409)
        self.counters["replayed"] += 1
        return {"replayed": True, "request": self._public(rec)}

    def submit(self, body: dict, key: str) -> dict:
        received = self.clock()  # JSON已解析；此后含准备、排队和后端，不含网络/读HTTP正文。
        body = validate_request(body)
        require(type(key) is str and re.fullmatch(r"[A-Za-z0-9_-]{1,64}", key) is not None,
                "idempotency_key", "需要1–64位字母数字下划线或连字符的 Idempotency-Key")
        body_hash = digest(body)
        with self.cv:
            self.counters["valid_submissions"] += 1
            self._sweep()
            old = self._existing(key, body_hash)
            if old is not None:
                return old
            require(self.accepting, "not_ready", "服务已停止接收新请求", 503)
        with self.prepare_lock:
            prepared = self.backend.prepare(body["prompt"])
        count = prepared.get("input_units")
        require(type(count) is int and count > 0, "backend_contract", "后端输入计数无效", 500)
        cost = count + body["max_new_tokens"]
        require(cost <= self.limits.max_context_units, "context_limit",
                "完整模板输入＋最大新增超过本课上下文上限；不会截断", 422)
        deadline = received + body["timeout_ms"] / 1000
        with self.cv:
            self._sweep()
            old = self._existing(key, body_hash)  # 并发同键时第二次核对
            if old is not None:
                return old
            require(self.accepting, "not_ready", "服务停止接收", 503)
            require(self.clock() < deadline, "deadline_before_admission", "准备输入期间已过期", 408)
            require(len(self.records) < self.limits.max_records, "ledger_full",
                    "本进程记录配额已满；本课不驱逐幂等记录，需要新服务实例", 503)
            if len(self.pending) >= self.limits.max_pending:
                self.counters["rejected_queue"] += 1
                raise ServiceError(503, "queue_full", "等待队列已满；请退避", True)
            if self.reserved + cost > self.limits.max_reserved_units:
                self.counters["rejected_reservation"] += 1
                raise ServiceError(503, "reservation_full", "累计预留单位预算已满", True)
            rid = uuid.uuid4().hex
            rec = RequestRecord(rid, key, body_hash, prepared, body["max_new_tokens"],
                                received, deadline, cost)
            self.records[rid] = rec
            self.keys[key] = rid
            self.pending.append(rid)
            self.reserved += cost
            self.counters["accepted"] += 1
            self._event(rec, "queued")
            self.cv.notify_all()
            return {"replayed": False, "request": self._public(rec)}

    def _public(self, rec: RequestRecord) -> dict:
        end = rec.finished
        start = rec.started
        ms = lambda a, b: None if a is None or b is None else max(0.0, (b-a)*1000)
        return {"request_id": rec.request_id, "service_epoch": self.epoch,
                "state": rec.state, "terminal": rec.state in TERMINAL,
                "input_units": rec.prepared["input_units"], "max_new_units": rec.max_new,
                "unit": self.backend.unit, "generated_units": rec.generated,
                "reserved_units": rec.reserved if rec.state not in TERMINAL else 0,
                "abort_reason": rec.abort_reason, "result": copy.deepcopy(rec.result),
                "error": copy.deepcopy(rec.error), "task_success": None,
                "timing": {"pre_start_ms": ms(rec.received, start if start is not None else end),
                           "service_ms": ms(start, end), "server_total_ms": ms(rec.received, end),
                           "server_first_output_ms": ms(rec.received, rec.first_output),
                           "cancel_is_cooperative": True},
                "events": copy.deepcopy(rec.events)}

    def get(self, rid: str) -> dict:
        with self.cv:
            self._sweep()
            require(rid in self.records, "not_found", "本服务实例没有该请求", 404)
            return self._public(self.records[rid])

    def cancel(self, rid: str) -> dict:
        with self.cv:
            self._sweep()
            require(rid in self.records, "not_found", "本服务实例没有该请求", 404)
            self._abort(self.records[rid], "client_cancel")
            self._sweep()
            self.cv.notify_all()
            return self._public(self.records[rid])

    def _stop_reason(self, rec):
        with self.cv:
            if self.clock() >= rec.deadline:
                self._abort(rec, "deadline")
            return rec.abort_reason

    def _observe(self, rec, count):
        with self.cv:
            require(type(count) is int and rec.generated <= count <= rec.max_new,
                    "backend_contract", "后端输出计数越界或倒退", 500)
            if count > 0 and rec.first_output is None:
                rec.first_output = self.clock()
            rec.generated = count

    def _work(self):
        while True:
            with self.cv:
                self._sweep()
                while not self.pending and not self.closed:
                    self.cv.wait(0.02)
                    self._sweep()
                if self.closed and not self.pending:
                    return
                rid = self.pending.popleft()
                rec = self.records[rid]
                if rec.state != "queued":
                    continue
                self.active = rid
                rec.started = self.clock()
                self._event(rec, "running")
                self.counters["backend_started"] += 1
            result, error = None, None
            try:
                result = self.backend.generate(
                    rec.prepared, rec.max_new,
                    lambda: self._stop_reason(rec), lambda n: self._observe(rec, n))
                require(type(result) is dict and type(result.get("text")) is str,
                        "backend_contract", "后端返回格式不合法", 500)
                require(type(result.get("generated_units")) is int
                        and 0 <= result["generated_units"] <= rec.max_new,
                        "backend_contract", "后端输出越过上限", 500)
                require(result["generated_units"] == rec.generated,
                        "backend_contract", "后端计数与观察不一致", 500)
                require(result.get("finish_reason") in {"eos", "length", "demo_end", "stopped"},
                        "backend_contract", "后端停止原因未知", 500)
                require(len(result["text"].encode("utf-8")) <= 32768,
                        "backend_contract", "响应文本过大", 500)
            except Exception as exc:
                # 不把prompt或堆栈/秘密直接放到HTTP错误中。
                error = {"code": "backend_error", "exception_type": type(exc).__name__}
                result = None
            with self.cv:
                self._stop_reason(rec)  # 后端未及时响应停止也不能被算作成功
                if rec.abort_reason:
                    terminal = "expired" if rec.abort_reason == "deadline" else "cancelled"
                else:
                    terminal = "failed" if error else "succeeded"
                self._finish(rec, terminal, result, error)
                self.active = None
                self.cv.notify_all()

    def _timer(self):
        while True:
            with self.cv:
                if self.closed:
                    return
                self._sweep()
                self.cv.wait(0.01)

    def metrics(self):
        with self.cv:
            self._sweep()
            return {"service_epoch": self.epoch, "accepting": self.accepting,
                    "worker_alive": self.worker.is_alive(), "active": int(self.active is not None),
                    "pending": len(self.pending), "reserved_units": self.reserved,
                    "records": len(self.records), "limits": vars(self.limits).copy(),
                    "backend": self.backend.metadata(), "counters": dict(self.counters),
                    "durable": False, "inference_concurrency": 1,
                    "reservations_are_vram_bytes": False}

    def wait(self, rid: str, seconds: float = 3) -> dict:
        until = time.monotonic() + seconds
        while True:
            state = self.get(rid)
            if state["terminal"]:
                return state
            if time.monotonic() >= until:
                raise TimeoutError("调用者等待结束；服务请求并未因此取消")
            time.sleep(0.002)

    def close(self, timeout: float = 5) -> dict:
        with self.cv:
            self.accepting = False
            self.closed = True
            for rec in list(self.records.values()):
                self._abort(rec, "shutdown")
            self._sweep()
            self.cv.notify_all()
        self.worker.join(timeout)
        self.timer.join(1)
        return {"worker_stopped": not self.worker.is_alive(),
                "hard_cancel_supported": False}

### 读控制器源码时，先追这五处

`submit()`：严格校验、幂等核对、输入准备、预留、排队。  
`_work()`：唯一推理 worker，处理一条请求后再取下一条。  
`_abort()`：排队请求直接结束；运行请求只进入 `stopping`。  
`_finish()`：只有这里统一释放预留，清除不再需要的输入状态。  
`_timer()`：后端忙时仍然扫描截止时间，不让过期队列一直占空间。

`Condition` 保护共享字典、队列与计数。并发请求不能只依赖“检查了一次长度”，必须在同一个受保护区域完成检查与预留。
这里还用了受控账本上限128：到上限以后拒绝新键，不静默驱逐旧键造成意外重复执行。

In [7]:
engine = RequestEngine(DemoBackend(step_delay=0))
try:
    accepted = engine.submit(request, "first-request")
    request_id = accepted["request"]["request_id"]
    finished = engine.wait(request_id)
    print("最终状态：", finished["state"])
    print("停止原因：", finished["result"]["finish_reason"])
    print("输出：", finished["result"]["text"])
    print("任务语义成功：", finished["task_success"])
    print("剩余预留：", engine.metrics()["reserved_units"])
finally:
    print("关闭：", engine.close())

最终状态： succeeded
停止原因： demo_end
输出： 演示：请求已经按预算完成。
任务语义成功： None
剩余预留： 0
关闭： {'worker_stopped': True, 'hard_cancel_supported': False}


## 4. 确认丢失后的重试：同一键不能对应不同请求

客户端没收到 `202` 时，服务端可能已经接受了请求。换一个新键重新提交，可能重复推理。
本课要求 `Idempotency-Key`：同键同内容返回同一个 request ID；同键不同内容或预算返回 `409`。

这里只保证**同一服务进程、结果未丢失**时不重复推理。没有持久化、跨进程锁或跨实例去重。
重启会产生新的 `service_epoch`，旧ID返回404；不可据此判定旧任务从未执行，更不能自动重试带副作用的工具。
第44课的持久化与未知效果核对仍然需要单独设计。

In [8]:
engine = RequestEngine(DemoBackend(step_delay=0))
try:
    first = engine.submit(request, "stable-key")
    engine.wait(first["request"]["request_id"])
    repeated = engine.submit(request, "stable-key")
    print("同一个ID：", first["request"]["request_id"] == repeated["request"]["request_id"])
    print("复用原记录：", repeated["replayed"])
    print("后端实际启动：", engine.metrics()["counters"]["backend_started"])
    try:
        engine.submit(dict(request, prompt="换一个问题"), "stable-key")
    except ServiceError as error:
        print("同键变内容：", error.status, error.code)
finally:
    engine.close()

同一个ID： True
复用原记录： True
后端实际启动： 1
同键变内容： 409 key_conflict


## 5. 加上真正的 HTTP 边界

服务只绑定回环地址。每次启动生成随机 Bearer 凭证；除了 `/healthz`，其余接口要求凭证。
本课拒绝 Origin、非预期Host、重复关键头、过大请求体，以及不支持的分块请求体。

这些限制是教学防线，**不是TLS、多用户鉴权、配额隔离或完整Web安全防护**。
不要把端口映射到公网。`http.server` 官方明确不建议用于生产。

HTTP回复写失败不会自动撤回已接收请求；轮询等待超时也不会隐式取消。

In [9]:
class LocalHTTPServer(ThreadingHTTPServer):
    daemon_threads = True
    request_queue_size = 8

    def __init__(self, address, handler, engine, api_key):
        require(address[0] == "127.0.0.1", "host", "仅绑定127.0.0.1")
        self.engine, self.api_key = engine, api_key
        self.slots = threading.BoundedSemaphore(8)
        super().__init__(address, handler)

    def get_request(self):
        sock, addr = super().get_request()
        sock.settimeout(3)
        return sock, addr

    def process_request(self, request, client_address):
        if not self.slots.acquire(blocking=False):
            body = b'{"error":{"code":"http_busy"}}'
            try:
                request.sendall(b"HTTP/1.1 503 Service Unavailable\r\nConnection: close\r\n"
                                b"Content-Type: application/json\r\nRetry-After: 1\r\nContent-Length: "
                                + str(len(body)).encode() + b"\r\n\r\n" + body)
            except OSError:
                pass
            self.shutdown_request(request)
            return
        try:
            super().process_request(request, client_address)
        except Exception:
            self.slots.release()
            raise

    def process_request_thread(self, request, client_address):
        try:
            super().process_request_thread(request, client_address)
        finally:
            self.slots.release()


class RequestHandler(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"
    server_version = "Lesson48Local"
    sys_version = ""

    def log_message(self, *_):
        pass  # 不打印 Authorization、prompt 或用户数据

    def _send(self, status, value, extra=None):
        raw = canonical(value).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(raw)))
        self.send_header("Connection", "close")
        self.send_header("Cache-Control", "no-store")
        for key, val in (extra or {}).items():
            self.send_header(key, str(val))
        self.end_headers()
        self.close_connection = True
        try:
            self.wfile.write(raw)
        except (BrokenPipeError, ConnectionResetError, TimeoutError):
            # 写回失败不意味着请求没被接受，也不意味着运行已经取消。
            pass

    def _guard(self):
        expected = f"127.0.0.1:{self.server.server_port}"
        hosts = self.headers.get_all("Host") or []
        require(len(hosts) == 1 and hosts[0] == expected, "host", "Host不匹配", 403)
        require(self.headers.get("Origin") is None, "origin", "本课不接受浏览器跨源请求", 403)
        require(len(self.path) <= 256 and "?" not in self.path and "#" not in self.path,
                "path", "不支持该路径")
        if self.path == "/healthz" and self.command == "GET":
            return
        auth = self.headers.get_all("Authorization") or []
        require(len(auth) == 1 and hmac.compare_digest(
            auth[0], "Bearer " + self.server.api_key), "unauthorized", "需要本机服务Bearer凭证", 401)

    def _body(self):
        require(self.headers.get("Transfer-Encoding") is None,
                "transfer_encoding", "本课不支持分块请求体")
        lengths = self.headers.get_all("Content-Length") or []
        require(len(lengths) == 1 and lengths[0].isdigit(), "length", "需要唯一Content-Length", 411)
        length = int(lengths[0])
        require(0 < length <= 16384, "body_too_large", "请求体上限16384字节", 413)
        content_types = self.headers.get_all("Content-Type") or []
        require(len(content_types) == 1 and content_types[0].split(";")[0].strip().lower() == "application/json",
                "content_type", "仅接受application/json", 415)
        raw = self.rfile.read(length)
        require(len(raw) == length, "incomplete_body", "请求体没有读完整")
        try:
            return strict_json(raw.decode("utf-8"))
        except UnicodeDecodeError as exc:
            raise ServiceError(400, "utf8", "请求体不是UTF-8") from exc

    def _dispatch(self):
        self._guard()
        engine = self.server.engine
        if self.command == "GET" and self.path == "/healthz":
            return self._send(200, {"alive": True, "meaning": "HTTP进程能响应，不代表模型可用"})
        if self.command == "GET" and self.path in {"/readyz", "/v1/metrics"}:
            data = engine.metrics()
            status = 200 if self.path == "/v1/metrics" or data["accepting"] and data["worker_alive"] else 503
            return self._send(status, data)
        if self.command == "POST" and self.path == "/v1/requests":
            keys = self.headers.get_all("Idempotency-Key") or []
            require(len(keys) == 1, "idempotency_key", "需要唯一Idempotency-Key")
            response = engine.submit(self._body(), keys[0])
            status = 200 if response["request"]["terminal"] else 202
            return self._send(status, response, {"Location": "/v1/requests/" + response["request"]["request_id"]})
        match = re.fullmatch(r"/v1/requests/([0-9a-f]{32})(/cancel)?", self.path)
        if match:
            rid, action = match.groups()
            if self.command == "GET" and action is None:
                return self._send(200, engine.get(rid))
            if self.command == "POST" and action == "/cancel":
                require(self._body() == {}, "cancel_body", "取消请求体必须是空对象")
                result = engine.cancel(rid)
                return self._send(200 if result["terminal"] else 202, result)
        raise ServiceError(404, "not_found", "没有这个接口")

    def _handle(self):
        try:
            self._dispatch()
        except ServiceError as exc:
            headers = {"Retry-After": "1"} if exc.retryable else None
            self._send(exc.status, {"error": {"code": exc.code, "message": str(exc),
                                               "retryable": exc.retryable}}, headers)
        except (TimeoutError, socket.timeout):
            self._send(408, {"error": {"code": "body_timeout"}})
        except (BrokenPipeError, ConnectionResetError):
            pass
        except Exception as exc:
            self._send(500, {"error": {"code": "internal_error", "exception_type": type(exc).__name__}})

    do_GET = _handle
    do_POST = _handle


@contextmanager
def local_service(backend=None, limits=None, port: int = 0, clock=time.monotonic):
    engine = RequestEngine(backend, limits, clock)
    server = None
    try:
        key = secrets.token_urlsafe(24)
        server = LocalHTTPServer(("127.0.0.1", port), RequestHandler, engine, key)
        thread = threading.Thread(target=server.serve_forever, kwargs={"poll_interval": 0.02}, daemon=True)
        thread.start()
        yield {"url": f"http://127.0.0.1:{server.server_port}", "api_key": key,
               "engine": engine, "server": server}
    finally:
        if server is not None:
            server.shutdown()
            server.server_close()
            thread.join(3)
        engine.close()

In [10]:
def http_call(url: str, method: str, path: str, api_key: str = "", body=None,
              idem_key: str | None = None, timeout: float = 3,
              raw_body: bytes | None = None, extra_headers=None) -> tuple[int, dict, dict]:
    parsed = urlsplit(url)
    require(parsed.scheme == "http" and parsed.hostname == "127.0.0.1" and parsed.port
            and parsed.path in {"", "/"} and not parsed.query and not parsed.fragment
            and parsed.username is None, "client_url", "客户端只访问显式本机回环端口")
    raw = raw_body if raw_body is not None else None if body is None else canonical(body).encode("utf-8")
    headers = {"Host": f"127.0.0.1:{parsed.port}", "Authorization": "Bearer " + api_key}
    if raw is not None:
        headers.update({"Content-Type": "application/json", "Content-Length": str(len(raw))})
    if idem_key is not None:
        headers["Idempotency-Key"] = idem_key
    headers.update(extra_headers or {})
    conn = http.client.HTTPConnection("127.0.0.1", parsed.port, timeout=timeout)
    try:
        conn.request(method, path, body=raw, headers=headers)
        response = conn.getresponse()
        data = response.read()
        return response.status, strict_json(data.decode("utf-8")), dict(response.getheaders())
    finally:
        conn.close()


def poll_http(url, api_key, rid, wait_seconds=5, interval=0.01):
    start = time.monotonic()
    while True:
        status, value, _ = http_call(url, "GET", "/v1/requests/" + rid, api_key)
        if status != 200:
            raise ServiceError(status, "poll", "获取请求失败")
        if value["terminal"]:
            return {"client_wait_state": "terminal_observed", "request": value,
                    "poll_wait_ms": (time.monotonic()-start)*1000}
        if time.monotonic() - start >= wait_seconds:
            return {"client_wait_state": "wait_timeout", "request": value,
                    "server_cancelled_by_this_timeout": False,
                    "poll_wait_ms": (time.monotonic()-start)*1000}
        time.sleep(interval)


def submit_and_wait(url, api_key, body, key, wait_seconds=10):
    start = time.monotonic()
    status, accepted, _ = http_call(url, "POST", "/v1/requests", api_key, body, key)
    if status not in {200, 202}:
        return {"http_status": status, "submission": accepted,
                "client_elapsed_ms": (time.monotonic()-start)*1000}
    rest = poll_http(url, api_key, accepted["request"]["request_id"], wait_seconds)
    return {"http_status": status, "submission": accepted, **rest,
            "client_elapsed_ms": (time.monotonic()-start)*1000}


def summarize(records: list[dict], planned: int) -> dict:
    require(type(planned) is int and planned > 0 and len(records) <= planned,
            "planned", "需要固定计划数量")
    ids = [r["request_id"] for r in records]
    require(len(set(ids)) == len(ids), "duplicate", "重复结果不能重复计数")
    for rec in records:
        require(rec["state"] in STATES, "state", "未知状态")
    counts = Counter(r["state"] for r in records)
    success = counts["succeeded"]
    completed_times = sorted(r["timing"]["server_total_ms"] for r in records
                             if r["terminal"] and r["timing"]["server_total_ms"] is not None)
    # 最近秩定义，避免把小样本插值结果包装为可靠尾延迟估计。
    p95 = completed_times[max(0, math.ceil(.95*len(completed_times))-1)] if completed_times else None
    return {"planned": planned, "received": len(records), "missing": planned-len(records),
            "statuses": dict(counts), "execution_success_fraction": success/planned,
            "task_success_fraction": None, "observed_terminal_p95_ms": p95,
            "latency_sample_count": len(completed_times),
            "p95_is_production_estimate": False}

In [11]:
with local_service(DemoBackend(0.002)) as service:
    url, api_key = service["url"], service["api_key"]
    status, alive, _ = http_call(url, "GET", "/healthz")
    print("存活：", status, alive)
    status, accepted, headers = http_call(
        url, "POST", "/v1/requests", api_key, request, "http-first")
    print("提交HTTP状态：", status)
    print("结果查询地址：", headers["Location"])
    observed = poll_http(url, api_key, accepted["request"]["request_id"])
    print("最终状态：", observed["request"]["state"])
    print("结果：", observed["request"]["result"])
# with退出后关闭HTTP与worker；不打印或保存api_key。

存活： 200 {'alive': True, 'meaning': 'HTTP进程能响应，不代表模型可用'}
提交HTTP状态： 202
结果查询地址： /v1/requests/1db42b52b58f4d3cb8b56fc53614a959
最终状态： succeeded
结果： {'finish_reason': 'demo_end', 'generated_units': 6, 'text': '演示：请求已经按预算完成。'}


## 6. 有界队列、取消与过期：别只测正常请求

下面用同步事件让第一条请求暂时停留在后端，然后向队列添加两条等待请求。
第四条不能继续排队，返回503。之后分别取消排队请求、等待另一个请求过期，再取消运行中的请求。

`GateBackend` 人为模拟无法立刻中断的一段计算；这是工程机制实验，不是故障GPU实测。
排队过期请求从未进入后端。取消运行中的请求后，先观察 `stopping`，再释放测试事件，让后端正常返回。

In [12]:
class GateBackend(DemoBackend):
    """只供测试：第一段模拟正在执行的不可中断块，release 后才检查取消。"""
    def __init__(self):
        super().__init__(0)
        self.entered, self.release = threading.Event(), threading.Event()
        self.started_calls = 0

    def generate(self, prepared, max_new, stop, observe):
        self.started_calls += 1
        if self.started_calls == 1:
            self.entered.set()
            if not self.release.wait(4):
                raise RuntimeError("测试没有释放gate")
        return super().generate(prepared, max_new, stop, observe)


class ManualClock:
    def __init__(self):
        self.value = 1000.0
        self.lock = threading.Lock()
    def __call__(self):
        with self.lock:
            return self.value
    def advance(self, seconds):
        with self.lock:
            self.value += seconds


def sample_request(prompt="你好", n=8, ms=5000):
    return {"prompt": prompt, "max_new_tokens": n, "timeout_ms": ms}


def wait_event(event):
    if not event.wait(2):
        raise AssertionError("测试事件未发生")


def lifecycle_demo() -> dict:
    """实际HTTP。事件控制后端暂停，稳定观察排队、拒绝和取消。"""
    gate = GateBackend()
    with local_service(gate, Limits(max_pending=2)) as svc:
        url, key, engine = svc["url"], svc["api_key"], svc["engine"]
        post = lambda b, k: http_call(url, "POST", "/v1/requests", key, b, k)
        s1, a, _ = post(sample_request(), "demo-a")
        rid_a = a["request"]["request_id"]
        wait_event(gate.entered)
        s2, b, _ = post(sample_request("排队取消"), "demo-b")
        s3, c, _ = post(sample_request("排队到期", ms=50), "demo-c")
        s4, full, _ = post(sample_request("不能无限排队"), "demo-full")
        same_status, same, _ = post(sample_request(), "demo-a")
        conflict, _, _ = post(sample_request("不同内容"), "demo-a")
        queued_cancel_status, cancelled, _ = http_call(
            url, "POST", "/v1/requests/" + b["request"]["request_id"] + "/cancel", key, {})
        expired = engine.wait(c["request"]["request_id"])
        active_cancel_status, cancelling, _ = http_call(
            url, "POST", "/v1/requests/" + rid_a + "/cancel", key, {})
        slot_held = engine.metrics()["active"] == 1 and engine.metrics()["reserved_units"] > 0
        gate.release.set()
        active_done = engine.wait(rid_a)
        good = submit_and_wait(url, key, sample_request("正常结束"), "demo-good")
        return {
            "http_statuses": {"first": s1, "queued": s2, "queue_full": s4,
                              "same_key": same_status, "conflict": conflict,
                              "queued_cancel": queued_cancel_status, "running_cancel": active_cancel_status},
            "same_key_same_request": same["request"]["request_id"] == rid_a,
            "queue_full_code": full["error"]["code"],
            "cancel_requested_state": cancelling["state"],
            "active_slot_held_until_backend_returns": slot_held,
            "queued_cancel_result": cancelled, "queued_expiry_result": expired,
            "running_cancel_result": active_done, "normal_result": good,
            "backend_started": engine.metrics()["counters"]["backend_started"],
            "metrics": engine.metrics(), "is_model_evaluation": False,
        }


def latency_demo() -> dict:
    # 真实回环HTTP时间 + 固定模拟工作量；不报告为模型tokens/s。
    results = []
    with local_service(DemoBackend(0.008), Limits(max_pending=4)) as svc:
        start = time.monotonic()
        accepted = []
        for i in range(4):
            status, body, _ = http_call(svc["url"], "POST", "/v1/requests", svc["api_key"],
                                       sample_request("固定演示工作量"), f"latency-{i}")
            require(status == 202, "demo", "演示未接受请求", 500)
            accepted.append(body["request"]["request_id"])
        for rid in accepted:
            observed = poll_http(svc["url"], svc["api_key"], rid)
            results.append(observed["request"])
        return {"records": results, "summary": summarize(results, planned=4),
                "batch_client_elapsed_ms": (time.monotonic()-start)*1000,
                "backend": svc["engine"].backend.metadata(),
                "scope": "4条真实回环HTTP请求，单worker，人工sleep；不测GPU或LLM性能"}


def budget_demo() -> dict:
    clock = ManualClock()
    budget = WorkBudget(1000, 2, 1, 100, clock)
    first = budget.claim("model", 10, 20)
    clock.advance(.75)
    second = budget.claim("tool")
    clock.advance(.30)
    try:
        budget.claim("model", 10, 20)
    except ServiceError as exc:
        error = exc.code
    else:
        raise AssertionError("截止预算错误重置")
    return {"first": first, "tool_after_750ms": second,
            "after_total_1050ms": error, "uses_manual_clock": True,
            "not_measured_backend_latency": True}

In [13]:
lifecycle = lifecycle_demo()
print("HTTP：", lifecycle["http_statuses"])
print("同键仍是同一请求：", lifecycle["same_key_same_request"])
print("运行中取消的即时状态：", lifecycle["cancel_requested_state"])
print("实际退出前保留执行槽：", lifecycle["active_slot_held_until_backend_returns"])
print("最终状态：", {
    "排队取消": lifecycle["queued_cancel_result"]["state"],
    "排队过期": lifecycle["queued_expiry_result"]["state"],
    "运行中取消": lifecycle["running_cancel_result"]["state"],
    "正常": lifecycle["normal_result"]["request"]["state"],
})
print("后端启动次数：", lifecycle["backend_started"], "（不是4次）")

HTTP： {'first': 202, 'queued': 202, 'queue_full': 503, 'same_key': 202, 'conflict': 409, 'queued_cancel': 200, 'running_cancel': 202}
同键仍是同一请求： True
运行中取消的即时状态： stopping
实际退出前保留执行槽： True
最终状态： {'排队取消': 'cancelled', '排队过期': 'expired', '运行中取消': 'cancelled', '正常': 'succeeded'}
后端启动次数： 2 （不是4次）


### 客户端停止等待，不等于服务器已经停止

客户端可以主动发送 `/cancel`，然后继续查看是否到达 `cancelled`。
如果仅仅关闭终端或HTTP连接，本课不会推断它要取消已经登记的工作。

这个约定有代价：客户端消失后，请求可能仍消耗资源，所以服务必须有自己的截止时间。
反过来，服务收到取消也不能假装正在运行的计算已经结束。

对于 Agent：取消请求不能撤销之前已经发送的邮件或已提交的付款。执行效果未知时，继续使用第44课的核对策略。
本课HTTP接口没有外部工具执行能力。

In [14]:
gate = GateBackend()
with local_service(gate) as service:
    try:
        status, accepted, _ = http_call(service["url"], "POST", "/v1/requests",
            service["api_key"], request, "client-timeout")
        rid = accepted["request"]["request_id"]
        wait_event(gate.entered)
        client_view = poll_http(service["url"], service["api_key"], rid, wait_seconds=0)
        print("客户端：", client_view["client_wait_state"])
        print("服务器：", service["engine"].get(rid)["state"])
        http_call(service["url"], "POST", "/v1/requests/"+rid+"/cancel", service["api_key"], {})
        print("显式取消后：", service["engine"].get(rid)["state"])
    finally:
        gate.release.set()
    print("后端退出后：", service["engine"].wait(rid)["state"])

客户端： wait_timeout
服务器： running
显式取消后： stopping
后端退出后： cancelled


## 7. 延迟分解：排队越长，不等于模型算得越慢

本课时间口径：

- `pre_start_ms`：已解析请求进入控制器到后端开始，包括tokenizer准备与排队，不包括HTTP收正文。
- `service_ms`：后端开始到真正返回，包括本课的生成、解码与观察开销。
- `server_total_ms`：上面两段之和；排队即终止的请求没有 `service_ms`。
- `server_first_output_ms`：服务端第一次观察到生成单位的时间。**不是客户端首字延迟**，因为本课没有流式返回。
- `client_elapsed_ms`：客户端从提交到最终轮询看到结果，包括网络、HTTP处理和轮询间隔。

下面真的执行四条回环HTTP请求，但每条后端只是六次人工sleep。这个实验是排队观察，不是模型tokens/s基准。

In [15]:
latency = latency_demo()
print("序号   启动前ms    后端ms      服务总ms")
for i, record in enumerate(latency["records"], 1):
    t = record["timing"]
    print(f"{i:2d}   {t['pre_start_ms']:10.3f} {t['service_ms']:10.3f} {t['server_total_ms']:10.3f}")
print("固定计划汇总：", latency["summary"])

序号   启动前ms    后端ms      服务总ms
 1        0.509     48.631     49.140
 2       47.996     49.694     97.691
 3       96.650     48.836    145.485
 4      144.599     48.830    193.428
固定计划汇总： {'planned': 4, 'received': 4, 'missing': 0, 'statuses': {'succeeded': 4}, 'execution_success_fraction': 1.0, 'task_success_fraction': None, 'observed_terminal_p95_ms': 193.42836199996327, 'latency_sample_count': 4, 'p95_is_production_estimate': False}


### 小样本 p95 与吞吐量的限制

本课用最近秩定义p95，并保存样本数。在4条样本上，它等于观测最大值，**不能当作可靠的生产尾延迟估计**。
没有做持续到达负载、长时间测量、冷启动对照或GPU显存基准；不能推导3060的服务吞吐上限。

同样，完成状态只统计执行：`succeeded` 不等于内容正确。
丢失或失败请求不能从计划分母中消失；同时要单独记录缺失原因，不把基础设施缺失混成模型答错。

In [16]:
# 假设计划4个请求，但只保存了前3个完成记录。
partial_summary = summarize(latency["records"][:3], planned=4)
print("执行成功比例：", partial_summary["execution_success_fraction"])
print("缺失：", partial_summary["missing"])
print("任务语义成功：", partial_summary["task_success_fraction"])

执行成功比例： 0.75
缺失： 1
任务语义成功： None


## 8. 接回 Agent：总预算跨越模型与工具边界

服务层限制每个模型请求，还不够。Agent可能连续创建很多模型请求和工具请求。

本课的 `WorkBudget` 示例在顶层维护：总截止时间、模型调用次数、工具调用次数与累计预留单位。
每个子步骤先领取额度，再把**剩余时限**传给下一层。失败与重试也消费预算，不通过重启子调用刷新它。

这些单位是保守预留，不是实际账单。真实多租户服务还需在可信宿主验证额度；不能相信模型在JSON中声称“还有100次”。

In [17]:
print(json.dumps(budget_demo(), ensure_ascii=False, indent=2))

{
  "first": {
    "remaining_ms": 1000,
    "model_calls": 1,
    "tool_calls": 0,
    "reserved_units": 30
  },
  "tool_after_750ms": {
    "remaining_ms": 250,
    "model_calls": 1,
    "tool_calls": 1,
    "reserved_units": 30
  },
  "after_total_1050ms": "deadline",
  "uses_manual_clock": true,
  "not_measured_backend_latency": true
}


## 9. 可选真实后端：固定Qwen、只加载一次、单worker调用

下面定义真实后端，但**定义类不会下载模型**。启动服务时才创建它，成功加载后才监听HTTP。
本课固定Qwen3-0.6B的revision，使用配套chat template、关闭thinking；完整模板长度在入队前检查。
[模型卡](https://huggingface.co/Qwen/Qwen3-0.6B/blob/c1899de289a04d12100db370d81485cdf75e47ca/README.md)

取消通过自定义 `StoppingCriteria` 请求停止，不保证正在执行的prefill或GPU算子立即退出。
[生成控制文档](https://huggingface.co/docs/transformers/v4.57.3/en/main_classes/text_generation)

每个请求独立使用KV cache；没有多用户共享聊天状态、连续批处理或paged attention。
内部streamer只记录生成计数并同步到CPU，**没有对HTTP客户端逐token推送**，也会影响性能。

In [18]:
class QwenBackend:
    name = "qwen3-0.6b"
    unit = "model_token"

    def __init__(self, device: str = "cpu", local_files_only: bool = False):
        try:
            import torch
            import transformers
            from transformers import AutoTokenizer, AutoModelForCausalLM
        except ImportError as exc:
            raise RuntimeError("真实后端需要 torch、transformers==4.57.3、safetensors；没有自动回退") from exc
        if transformers.__version__ != "4.57.3":
            raise RuntimeError("本课真实接口固定 transformers==4.57.3，请使用独立课程环境")
        if device == "cuda" and not torch.cuda.is_available():
            raise RuntimeError("明确请求了 CUDA，但当前 PyTorch 不可用")
        if device not in {"cpu", "cuda"}:
            raise ValueError("device 只能是 cpu 或 cuda")
        self.torch, self.device = torch, device
        self.dtype = torch.float16 if device == "cuda" else torch.float32
        self.tokenizer = AutoTokenizer.from_pretrained(
            MODEL_ID, revision=REVISION, trust_remote_code=False,
            local_files_only=local_files_only)
        self.model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, revision=REVISION, trust_remote_code=False,
            local_files_only=local_files_only, use_safetensors=True,
            torch_dtype=self.dtype).to(device).eval()
        raw_eos = self.model.generation_config.eos_token_id
        self.eos = [raw_eos] if type(raw_eos) is int else list(raw_eos or [])
        if not self.eos:
            raise RuntimeError("模型没有可用的 EOS 配置")
        self.pad_id = self.tokenizer.pad_token_id
        if self.pad_id is None:
            self.pad_id = self.eos[0]

    def prepare(self, prompt: str) -> dict:
        messages = [
            {"role": "system", "content": "你是一名中文助手。请直接、简洁地回答，不执行外部工具。"},
            {"role": "user", "content": prompt},
        ]
        ids = self.tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True, enable_thinking=False)
        return {"input_units": len(ids), "input_ids": ids}

    def generate(self, prepared: dict, max_new: int,
                 stop: Callable[[], str | None], observe: Callable[[int], None]) -> dict:
        from transformers import GenerationConfig, StoppingCriteria, StoppingCriteriaList
        from transformers.generation.streamers import BaseStreamer
        torch = self.torch

        class CooperativeStop(StoppingCriteria):
            def __call__(self, input_ids, scores, **kwargs):
                return torch.full((input_ids.shape[0],), bool(stop()),
                                  device=input_ids.device, dtype=torch.bool)

        class CounterStreamer(BaseStreamer):
            # 只记录生成进度，不通过 HTTP 流式输出。put 会发生 CPU 同步，影响性能。
            def __init__(self):
                self.prompt_seen = False
                self.count = 0
            def put(self, value):
                if not self.prompt_seen:
                    self.prompt_seen = True
                    return
                self.count += int(value.numel())
                observe(self.count)
            def end(self):
                pass

        tracker = CounterStreamer()
        ids = torch.tensor([prepared["input_ids"]], dtype=torch.long, device=self.device)
        mask = torch.ones_like(ids)
        if stop():
            return {"text": "", "generated_units": 0, "finish_reason": "stopped"}
        config = GenerationConfig(
            max_new_tokens=max_new, do_sample=False, num_beams=1, use_cache=True,
            eos_token_id=self.eos, pad_token_id=self.pad_id)
        with torch.inference_mode():
            output = self.model.generate(
                input_ids=ids, attention_mask=mask, generation_config=config,
                stopping_criteria=StoppingCriteriaList([CooperativeStop()]), streamer=tracker)
            new_ids = output[0, ids.shape[1]:].detach().cpu().tolist()
        # detach/cpu 已等待输出就绪；若 CUDA 内核挂起，本函数仍可能无法及时返回。
        text = self.tokenizer.decode(new_ids, skip_special_tokens=True)
        return {"text": text, "generated_units": len(new_ids),
                "finish_reason": "eos" if new_ids and new_ids[-1] in self.eos else "length"}

    def metadata(self) -> dict:
        import transformers
        return {"name": self.name, "unit": self.unit, "is_language_model": True,
                "model_id": MODEL_ID, "revision": REVISION, "device": self.device,
                "dtype": str(self.dtype), "transformers": transformers.__version__,
                "torch": self.torch.__version__, "batched": False,
                "performance_benchmark": False}

In [19]:
RUN_REAL_SERVICE = False
REAL_DEVICE = "cpu"   # 已配置3060后可显式改成cuda，不会自动回退
LOCAL_FILES_ONLY = False
if RUN_REAL_SERVICE:
    # 只完成一次请求，然后with自动关闭服务；不是永久服务单元。
    real_backend = QwenBackend(REAL_DEVICE, LOCAL_FILES_ONLY)
    with local_service(real_backend) as service:
        real_result = submit_and_wait(
            service["url"], service["api_key"],
            {"prompt":"用一句话解释KV cache。", "max_new_tokens":64, "timeout_ms":120000},
            "real-notebook-request", wait_seconds=125)
        print(json.dumps(real_result, ensure_ascii=False, indent=2))
else:
    print("真实Qwen单元未开启：没有下载、没有模型回答、没有GPU测量。")

真实Qwen单元未开启：没有下载、没有模型回答、没有GPU测量。


### 安装与真实验证边界

沿用已有可用的PyTorch环境，本课接口固定为：

```bash
python -m pip install "transformers==4.57.3" safetensors
```

这不是最新版本声明，也不是已在制作环境验证的CUDA组合。代码明确检查课程接口版本，失败不会回退演示后端。

CPU使用FP32，CUDA使用FP16；没有量化、自动CPU卸载或峰值显存测量。模型、分词器和计算错误会明确记录/报错。
`max_context_units=1024` 是课堂服务上限，不代表模型官方最大上下文。

检查点、运行日志和身份认证是不同模块：服务只在内存中保存结果，没有实现第44课的持久工作流恢复。

## 10. 全套检查与落盘报告

这些检查实际启动回环HTTP服务器，不只是调用假的handler。包含并发同键、错误输入、排队到期、运行取消、
执行槽保留、原结果复用、服务重启、客户端停止等待、错误计数等。

结果账本只服务单一实例。同一API密钥可查看该实例内已知ID，不提供多用户隔离。不要把它连接到生产数据。

In [20]:
def run_checks() -> dict:
    checks = []
    def check(name, condition):
        if not condition:
            raise AssertionError(name)
        checks.append(name)
    def rejects(name, fn, code):
        try:
            fn()
        except ServiceError as exc:
            check(name, exc.code == code)
        else:
            raise AssertionError(name + " 未拒绝")

    check("正常请求", validate_request(sample_request()) == sample_request())
    for field, value, code in [("max_new_tokens", True, "output_limit"),
                               ("max_new_tokens", 0, "output_limit"),
                               ("max_new_tokens", 257, "output_limit"),
                               ("timeout_ms", True, "timeout"), ("timeout_ms", 24, "timeout"),
                               ("timeout_ms", 120001, "timeout"), ("prompt", " ", "prompt"),
                               ("prompt", 42, "prompt"), ("prompt", "中"*1400, "prompt_too_large")]:
        data = sample_request(); data[field] = value
        rejects("拒绝字段 " + field + repr(value)[:25], lambda d=data: validate_request(d), code)
    rejects("拒绝自授权字段", lambda: validate_request({**sample_request(), "approved": True}), "schema")
    rejects("重复JSON键", lambda: strict_json('{"a":1,"a":2}'), "duplicate_key")
    rejects("NaN", lambda: strict_json('{"a":NaN}'), "nonfinite")
    rejects("指数溢出", lambda: strict_json('{"a":1e9999}'), "nonfinite")
    rejects("半个JSON", lambda: strict_json('{"a":'), "invalid_json")
    clock = ManualClock(); budget = WorkBudget(1000, 1, 1, 10, clock)
    check("第一次模型预留", budget.claim("model", 3, 4)["reserved_units"] == 7)
    rejects("不允许额外模型调用", lambda: budget.claim("model", 0, 1), "model_budget")
    clock.advance(.75)
    check("子步骤共享截止时间", budget.claim("tool")["remaining_ms"] == 250)
    clock.advance(.25)
    rejects("恰好截止也过期", lambda: budget.claim("tool"), "deadline")
    b2 = WorkBudget(1000, 2, 1, 5)
    rejects("过度预留", lambda: b2.claim("model", 3, 3), "unit_budget")
    check("拒绝不消耗计数", b2.model_calls == b2.reserved_units == 0)
    rejects("负预算", lambda: WorkBudget(-1, 1, 1, 1), "budget")

    engine = RequestEngine(DemoBackend(0))
    try:
        a = engine.submit(sample_request(n=2), "a")
        result = engine.wait(a["request"]["request_id"])
        check("正常执行完成", result["state"] == "succeeded")
        check("输出长度受控", result["generated_units"] == 2 and result["result"]["finish_reason"] == "length")
        check("状态成功不冒充任务成功", result["task_success"] is None)
        check("单位明确", result["unit"] == "demo_codepoint_unit")
        check("计时非负", all(x is None or x >= 0 for k, x in result["timing"].items() if k.endswith("_ms")))
        check("预启动加运行等于总时间", abs(result["timing"]["pre_start_ms"] + result["timing"]["service_ms"] - result["timing"]["server_total_ms"]) < 1e-6)
        before = engine.metrics()
        replay = engine.submit(sample_request(n=2), "a")
        check("同键同请求", replay["replayed"] and replay["request"]["request_id"] == result["request_id"])
        check("幂等不重复调用", engine.metrics()["counters"]["backend_started"] == before["counters"]["backend_started"])
        rejects("同键改参数冲突", lambda: engine.submit(sample_request(n=3), "a"), "key_conflict")
        check("完成后取消不改状态", engine.cancel(result["request_id"])["state"] == "succeeded")
        check("最终预留释放", engine.metrics()["reserved_units"] == 0)
        rejects("超上下文不截断", lambda: engine.submit(sample_request("a"*1000, n=256), "large"), "context_limit")
        rejects("不认识的请求", lambda: engine.get("a"*32), "not_found")
        rejects("非法键", lambda: engine.submit(sample_request(), "bad key"), "idempotency_key")
        check("响应不泄露prompt", "prompt" not in result and "input_ids" not in result)
    finally:
        closed = engine.close()
    check("关闭worker", closed["worker_stopped"])
    rejects("关闭后不接新单", lambda: engine.submit(sample_request(), "after-close"), "not_ready")

    gate = GateBackend(); clock = ManualClock()
    engine = RequestEngine(gate, Limits(max_pending=2), clock)
    try:
        a = engine.submit(sample_request(ms=5000), "first")["request"]
        wait_event(gate.entered)
        b = engine.submit(sample_request(ms=100), "expire")["request"]
        c = engine.submit(sample_request(), "cancel")["request"]
        rejects("队列满执行前拒绝", lambda: engine.submit(sample_request(), "full"), "queue_full")
        clock.advance(.2)
        b_done = engine.get(b["request_id"])
        check("排队期间也消耗截止预算", b_done["state"] == "expired")
        check("过期排队请求从未开始", b_done["timing"]["service_ms"] is None and gate.started_calls == 1)
        check("取消排队请求立即终态", engine.cancel(c["request_id"])["state"] == "cancelled")
        reserved_before = engine.metrics()["reserved_units"]
        stopping = engine.cancel(a["request_id"])
        check("运行中取消仅请求停止", stopping["state"] == "stopping" and not stopping["terminal"])
        check("尚未结束不能释放运行槽", engine.metrics()["active"] == 1)
        check("尚未结束不能释放预留", engine.metrics()["reserved_units"] == reserved_before > 0)
        gate.release.set()
        done = engine.wait(a["request_id"])
        check("实际退出才取消", done["state"] == "cancelled")
        check("取消不能计为成功", done["result"]["generated_units"] == 0)
        check("最终无资源遗留", engine.metrics()["reserved_units"] == 0)
        check("未把排队时间重置", done["timing"]["server_total_ms"] >= 200-1e-6)
    finally:
        gate.release.set(); engine.close()

    gate = GateBackend(); clock = ManualClock()
    engine = RequestEngine(gate, clock=clock)
    try:
        rid = engine.submit(sample_request(ms=100), "active-deadline")["request"]["request_id"]
        wait_event(gate.entered); clock.advance(.15)
        check("运行时到期先stopping", engine.get(rid)["state"] == "stopping")
        check("到期未释放活跃槽", engine.metrics()["active"] == 1)
        gate.release.set()
        check("运行时到期终态expired", engine.wait(rid)["state"] == "expired")
    finally:
        gate.release.set(); engine.close()

    gate = GateBackend()
    engine = RequestEngine(gate, Limits(max_context_units=64, max_reserved_units=64, max_pending=3))
    try:
        rid = engine.submit(sample_request("A", n=32), "reserve")["request"]["request_id"]
        wait_event(gate.entered)
        rejects("累计预留限制独立于队列长度", lambda: engine.submit(sample_request("B", n=32), "over"), "reservation_full")
        gate.release.set(); engine.wait(rid)
        check("拒绝后同键可重新提交", not engine.submit(sample_request("B", n=32), "over")["replayed"])
    finally:
        gate.release.set(); engine.close()

    engine = RequestEngine(DemoBackend(0), Limits(max_records=1))
    try:
        rid = engine.submit(sample_request(), "one")["request"]["request_id"]; engine.wait(rid)
        rejects("结果账本有上限", lambda: engine.submit(sample_request(), "two"), "ledger_full")
        check("账本满仍可重取旧键", engine.submit(sample_request(), "one")["replayed"])
    finally:
        engine.close()

    class BrokenBackend(DemoBackend):
        def generate(self, *args):
            raise RuntimeError("敏感调试内容不进入API")
    engine = RequestEngine(BrokenBackend())
    try:
        r = engine.wait(engine.submit(sample_request(), "broken")["request"]["request_id"])
        check("异常记录failed", r["state"] == "failed")
        check("异常不泄露详细内容", "敏感" not in canonical(r))
        check("失败释放预留", engine.metrics()["reserved_units"] == 0)
    finally:
        engine.close()

    # 并发提交同键，只会建立一条请求。准备可重复，但不会重复推理。
    from concurrent.futures import ThreadPoolExecutor
    gate = GateBackend(); engine = RequestEngine(gate)
    try:
        with ThreadPoolExecutor(max_workers=4) as pool:
            values = list(pool.map(lambda _: engine.submit(sample_request(), "race"), range(4)))
        check("同键并发只建一条", len({v["request"]["request_id"] for v in values}) == 1)
        check("同键并发只有一次新建", sum(not v["replayed"] for v in values) == 1)
        gate.release.set(); engine.wait(values[0]["request"]["request_id"])
        check("同键并发后端一次", gate.calls == 1)
    finally:
        gate.release.set(); engine.close()

    gate = GateBackend()
    with local_service(gate) as svc:
        url, key = svc["url"], svc["api_key"]
        check("HTTP存活检查", http_call(url, "GET", "/healthz")[0] == 200)
        check("缺少凭证拒绝", http_call(url, "GET", "/readyz")[0] == 401)
        check("有凭证就绪检查", http_call(url, "GET", "/readyz", key)[0] == 200)
        check("限制Origin", http_call(url, "GET", "/readyz", key, extra_headers={"Origin":"http://x"})[0] == 403)
        check("限制Host", http_call(url, "GET", "/readyz", key, extra_headers={"Host":"evil.test"})[0] == 403)
        check("未知路由404", http_call(url, "GET", "/arbitrary", key)[0] == 404)
        status, a, headers = http_call(url, "POST", "/v1/requests", key, sample_request(), "http-a")
        check("接受返回202", status == 202 and "Location" in headers)
        check("202不是任务成功", not a["request"]["terminal"])
        rid = a["request"]["request_id"]
        wait_event(gate.entered)
        client_wait = poll_http(url, key, rid, wait_seconds=.001, interval=.001)
        check("客户等待超时不会自动取消", client_wait["client_wait_state"] == "wait_timeout" and svc["engine"].get(rid)["state"] == "running")
        check("HTTP重复请求仍同ID", http_call(url, "POST", "/v1/requests", key, sample_request(), "http-a")[1]["request"]["request_id"] == rid)
        check("HTTP键冲突409", http_call(url, "POST", "/v1/requests", key, sample_request("changed"), "http-a")[0] == 409)
        check("HTTP无幂等键400", http_call(url, "POST", "/v1/requests", key, sample_request())[0] == 400)
        check("HTTP重复JSON键400", http_call(url, "POST", "/v1/requests", key, idem_key="badjson", raw_body=b'{"a":1,"a":2}')[0] == 400)
        check("HTTP正文上限413", http_call(url, "POST", "/v1/requests", key, idem_key="huge", raw_body=b" "*17000)[0] == 413)
        check("HTTP媒体类型415", http_call(url, "POST", "/v1/requests", key, sample_request(), "type", extra_headers={"Content-Type":"text/plain"})[0] == 415)
        check("HTTP取消运行返回202", http_call(url, "POST", f"/v1/requests/{rid}/cancel", key, {})[0] == 202)
        gate.release.set()
        check("HTTP取消最终生效", poll_http(url, key, rid)["request"]["state"] == "cancelled")
        old_epoch = svc["engine"].epoch
    with local_service(DemoBackend(0)) as svc:
        check("重启会换服务身份", old_epoch != svc["engine"].epoch)
        check("重启旧ID明确未知", http_call(svc["url"], "GET", "/v1/requests/"+rid, svc["api_key"])[0] == 404)
    example = result
    s = summarize([example], 2)
    check("固定分母保留缺失", s["execution_success_fraction"] == .5 and s["missing"] == 1)
    check("执行成功不代表任务成功", s["task_success_fraction"] is None)
    rejects("重复记录不重复计分", lambda: summarize([example, example], 2), "duplicate")
    rejects("客户端不访问远程", lambda: http_call("http://example.com:80", "GET", "/"), "client_url")
    return {"count": len(checks), "passed": checks,
            "scope": "标准库单元检查＋真实回环HTTP，不是Qwen、GPU或生产压测"}

In [21]:
check_report = run_checks()
print("固定检查通过：", check_report["count"])
print(check_report["scope"])

固定检查通过： 86
标准库单元检查＋真实回环HTTP，不是Qwen、GPU或生产压测


In [22]:
def write_json(path: Path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False)+"\n", encoding="utf-8")
    os.replace(temporary, path)


def run_demo(output_root="lesson_48_outputs"):
    out = Path(output_root) / ("local_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "_" + uuid.uuid4().hex[:8])
    out.mkdir(parents=True, exist_ok=False)
    manifest = {"format": FORMAT, "python": sys.version, "platform": platform.platform(),
                "backend": "deterministic-demo", "real_model_tested": False,
                "cuda_tested": False, "production_load_test": False, "sources": SOURCES,
                "status": "running"}
    write_json(out/"manifest.json", manifest)
    try:
        checks = run_checks()
        lifecycle = lifecycle_demo()
        latency = latency_demo()
        budgets = budget_demo()
        for name, data in [("checks",checks),("lifecycle",lifecycle),("latency",latency),("budgets",budgets)]:
            write_json(out/(name+".json"), data)
        summary = {"checks_passed": checks["count"],
                   "protocol": lifecycle["http_statuses"],
                   "running_cancel_before_return": lifecycle["cancel_requested_state"],
                   "running_slot_not_released_early": lifecycle["active_slot_held_until_backend_returns"],
                   "states": {"cancelled_queued":lifecycle["queued_cancel_result"]["state"],
                              "expired_queued":lifecycle["queued_expiry_result"]["state"],
                              "cancelled_running":lifecycle["running_cancel_result"]["state"],
                              "normal":lifecycle["normal_result"]["request"]["state"]},
                   "latency_units": "实际回环HTTP毫秒；人工延时后端，不是模型性能",
                   "task_success": None}
        write_json(out/"summary.json", summary)
        manifest["status"] = "completed"
        write_json(out/"manifest.json", manifest)
        return {"directory": str(out), "summary": summary, "latency": latency}
    except Exception as exc:
        manifest.update(status="failed", exception_type=type(exc).__name__)
        write_json(out/"manifest.json", manifest)
        raise


def doctor():
    import importlib.util
    return {"python": sys.version, "platform": platform.platform(),
            "packages": {name: importlib.util.find_spec(name) is not None
                         for name in ("torch", "transformers", "safetensors")},
            "default_needs": "Python标准库；可创建本机回环socket", "sources": SOURCES}


def main(argv=None):
    p = argparse.ArgumentParser(description=__doc__)
    group = p.add_mutually_exclusive_group()
    group.add_argument("--checks-only", action="store_true")
    group.add_argument("--doctor", action="store_true")
    group.add_argument("--serve", action="store_true")
    group.add_argument("--client", metavar="URL")
    p.add_argument("--backend", choices=("demo","qwen"), default="demo")
    p.add_argument("--device", choices=("cpu","cuda"), default="cpu")
    p.add_argument("--local-files-only", action="store_true")
    p.add_argument("--port", type=int, default=8765)
    p.add_argument("--ready-file", type=Path)
    p.add_argument("--output-dir", default="lesson_48_outputs")
    p.add_argument("--prompt", default="用一句话解释KV cache。")
    p.add_argument("--max-new-tokens", type=int, default=64)
    p.add_argument("--timeout-ms", type=int, default=30000)
    p.add_argument("--wait-seconds", type=float, default=35)
    p.add_argument("--key", default=None, help="重试同一逻辑请求时复用；不同正文不可同键")
    p.add_argument("--request-id", default=None, help="客户端查看已知请求，不再提交")
    p.add_argument("--cancel", action="store_true", help="配合--client --request-id显式取消")
    args = p.parse_args(argv)
    if args.doctor:
        print(json.dumps(doctor(), ensure_ascii=False, indent=2)); return 0
    if args.checks_only:
        checks = run_checks()
        print(f"固定检查通过：{checks['count']}项。真实本地HTTP；不是Qwen/GPU或生产压测。")
        return 0
    if args.client:
        key = os.environ.get("LESSON48_API_KEY", "")
        require(bool(key), "api_key", "先设置环境变量LESSON48_API_KEY")
        if args.request_id:
            path = "/v1/requests/" + args.request_id
            value = http_call(args.client, "POST" if args.cancel else "GET",
                              path + ("/cancel" if args.cancel else ""), key,
                              {} if args.cancel else None)
        else:
            require(not args.cancel, "request_id", "取消需要request-id")
            value = submit_and_wait(args.client, key,
                    sample_request(args.prompt,args.max_new_tokens,args.timeout_ms),
                    args.key or uuid.uuid4().hex, args.wait_seconds)
        print(json.dumps(value, ensure_ascii=False, indent=2)); return 0
    if args.serve:
        require(0 <= args.port <= 65535, "port", "端口范围错误")
        backend = DemoBackend() if args.backend == "demo" else QwenBackend(args.device,args.local_files_only)
        # 真实模型加载成功后才启动HTTP，失败绝不换演示后端。
        with local_service(backend, port=args.port) as svc:
            info = {"url":svc["url"],"api_key":svc["api_key"],"backend":backend.metadata(),
                    "service_epoch":svc["engine"].epoch,"warning":"仅本机教学，Ctrl+C停止"}
            if args.ready_file:
                # 只存URL与身份，不把凭证写入报告；交互使用请从终端复制。
                write_json(args.ready_file,{k:v for k,v in info.items() if k != "api_key"})
            print(json.dumps(info, ensure_ascii=False, indent=2), flush=True)
            try:
                while True: time.sleep(.5)
            except KeyboardInterrupt:
                print("停止接收；向未完成请求发出停止信号。不是强制终止内核。", flush=True)
        return 0
    require(args.backend == "demo", "mode", "真实后端必须明确使用--serve --backend qwen")
    report = run_demo(args.output_dir)
    print(json.dumps({"directory":report["directory"], **report["summary"]}, ensure_ascii=False, indent=2))
    return 0

In [23]:
# 再跑一次默认实验并落盘，方便与独立脚本比较。实际时间和请求ID不要求逐位相同。
RUN_SAVED_EXPERIMENT = True
if RUN_SAVED_EXPERIMENT:
    saved = run_demo("lesson_48_outputs")
    print("输出：", saved["directory"])
    print(json.dumps(saved["summary"], ensure_ascii=False, indent=2))

输出： lesson_48_outputs/local_20261102T112147_c0ba1f3e
{
  "checks_passed": 86,
  "protocol": {
    "first": 202,
    "queued": 202,
    "queue_full": 503,
    "same_key": 202,
    "conflict": 409,
    "queued_cancel": 200,
    "running_cancel": 202
  },
  "running_cancel_before_return": "stopping",
  "running_slot_not_released_early": true,
  "states": {
    "cancelled_queued": "cancelled",
    "expired_queued": "expired",
    "cancelled_running": "cancelled",
    "normal": "succeeded"
  },
  "latency_units": "实际回环HTTP毫秒；人工延时后端，不是模型性能",
  "task_success": null
}


## 11. 独立练习：先完成，再看后面的答案

**A：跨步骤预算**。实现 `claim_model_ex()`，使用同一个预算对象领取输入＋最大输出，不重置截止时间，不漏掉调用次数。

**B：终态决定**。实现 `terminal_state_ex()`：已经到期/取消的请求，即使后端后来返回了文字，也不能标记成功。
这个函数只能在后端已经返回后调用，不能拿它提前释放执行槽。

**C：固定分母评测**。实现 `service_summary_ex()`，保留缺失请求的分母，同时明确任务语义正确性未判断。

运行全部单元不等于完成练习；三个开关默认关闭。

In [24]:
def budget_claim_reference(budget, input_units, max_new_units):
    return budget.claim("model", input_units, max_new_units)


def next_terminal_reference(abort_reason, backend_failed):
    require(abort_reason in {None, "deadline", "client_cancel", "shutdown"},
            "reason", "未知停止原因")
    require(type(backend_failed) is bool, "failed", "backend_failed需要布尔值")
    if abort_reason == "deadline":
        return "expired"
    if abort_reason in {"client_cancel", "shutdown"}:
        return "cancelled"
    return "failed" if backend_failed else "succeeded"


def fixed_denominator_reference(states, planned):
    require(type(planned) is int and planned > 0 and len(states) <= planned,
            "planned", "计划数量无效")
    require(all(state in STATES for state in states), "state", "未知状态")
    return {"verified_execution_success": sum(s == "succeeded" for s in states)/planned,
            "missing": planned-len(states), "task_success": None}


def check_exercise_a(fn):
    clock = ManualClock(); budget = WorkBudget(1000, 2, 1, 50, clock)
    a = fn(budget, 10, 10); clock.advance(.75)
    b = fn(budget, 10, 10)
    assert a["reserved_units"] == 20 and b["reserved_units"] == 40
    assert b["remaining_ms"] == 250
    try: fn(budget, 1, 1)
    except ServiceError as exc: assert exc.code == "model_budget"
    else: raise AssertionError("漏掉调用上限")
    return "A通过"


def check_exercise_b(fn):
    for reason, failed, expected in [(None,False,"succeeded"),(None,True,"failed"),
                                    ("client_cancel",False,"cancelled"),
                                    ("deadline",False,"expired"),("shutdown",True,"cancelled")]:
        assert fn(reason, failed) == expected
    return "B通过；只能在后端已经返回后使用这个终态选择函数"


def check_exercise_c(fn):
    value = fn(["succeeded","failed","expired"],4)
    assert value["verified_execution_success"] == .25 and value["missing"] == 1
    assert value["task_success"] is None
    try: fn(["not-a-state"],1)
    except ServiceError: pass
    else: raise AssertionError("漏掉状态校验")
    return "C通过"

In [25]:
def claim_model_ex(budget, input_units, max_new_units):
    raise NotImplementedError("请调用共享预算，不创建新预算")

def terminal_state_ex(abort_reason, backend_failed):
    raise NotImplementedError("后端返回以后决定终态")

def service_summary_ex(states, planned):
    raise NotImplementedError("用计划数量作分母，不把完成等同答对")

RUN_EXERCISE_A = False
RUN_EXERCISE_B = False
RUN_EXERCISE_C = False
if RUN_EXERCISE_A:
    print(check_exercise_a(claim_model_ex))
if RUN_EXERCISE_B:
    print(check_exercise_b(terminal_state_ex))
if RUN_EXERCISE_C:
    print(check_exercise_c(service_summary_ex))

## 12. 参考答案

A的关键是复用同一个对象中的截止时间与计数；B是优先处理停止信号；C保留计划分母。
不要为了让测试通过而把异常输入直接当成功，也不要把`stopping`算成最终取消。

In [26]:
# 与学生函数不同名，不覆盖你的实现。
print(check_exercise_a(budget_claim_reference))
print(check_exercise_b(next_terminal_reference))
print(check_exercise_c(fixed_denominator_reference))

A通过
B通过；只能在后端已经返回后使用这个终态选择函数
C通过


## 13. 在终端启动服务，并从另一个进程调用

**终端A：**

```bash
python lesson_48_inference_service_lifecycle_and_budgets.py --serve --port 8765
```

终端会显示URL、随机API凭证和后端身份。只监听本机。Ctrl+C停止接收，并请求未完成工作停止。

**终端B：**先设置凭证（替换实际值，不要把它提交到Git）。

Bash：
```bash
export LESSON48_API_KEY='复制终端A的api_key'
```
PowerShell：
```powershell
$env:LESSON48_API_KEY = '复制终端A的api_key'
```

然后提交：
```bash
python lesson_48_inference_service_lifecycle_and_budgets.py --client http://127.0.0.1:8765 --prompt "解释KV cache" --key request-001
```

只等待零秒可观察非终态；使用结果中的request_id查看或取消：
```bash
python lesson_48_inference_service_lifecycle_and_budgets.py --client http://127.0.0.1:8765 --request-id 实际ID
python lesson_48_inference_service_lifecycle_and_budgets.py --client http://127.0.0.1:8765 --request-id 实际ID --cancel
```

同一逻辑重试复用同一key与相同正文。改变输入、预算时换新key；已有结果仍使用旧key查询。
服务重启会换epoch与凭证，本课不承诺跨重启幂等。

**真实模型服务：**
```bash
python lesson_48_inference_service_lifecycle_and_budgets.py --serve --backend qwen --device cuda --port 8765
```

**默认完整实验与固定检查：**
```bash
python lesson_48_inference_service_lifecycle_and_budgets.py
python lesson_48_inference_service_lifecycle_and_budgets.py --checks-only
```

## 14. 验收与边界

实际完成一次提交、查询和取消；解释`queued`、`stopping`与终态。确认同键不会重复启动后端，变内容会冲突。
观察四条相同工作量的请求怎样产生不同排队时间。

本课已经实现：真实本机HTTP、输入校验、有界等待与预留、一个推理worker、协作取消、排队截止、实例内幂等、固定分母评测。

**尚未实现**：TLS、用户隔离、生产服务器、硬中断、跨进程持久化、SSE、连续批处理、分页KV、租户限流、公网部署或真实压力测试。
一次单元测试通过不是安全认证；四条请求也不能代表服务稳定性。

下一课：**第49课，流式输出与背压——SSE、增量解码、客户端断开与输出缓冲预算**。

## 主要资料（协议与官方文档）

- [HTTP语义，202与503、Retry-After](https://www.rfc-editor.org/rfc/rfc9110.html)
- [Python http.server 的教学边界](https://docs.python.org/3.13/library/http.server.html)
- [单调时钟](https://docs.python.org/3.13/library/time.html#time.monotonic)
- [Future取消限制](https://docs.python.org/3.13/library/concurrent.futures.html)
- [Transformers生成控制](https://huggingface.co/docs/transformers/v4.57.3/en/main_classes/text_generation)
- [停止条件与streamer](https://huggingface.co/docs/transformers/v4.57.3/en/internal/generation_utils)
- [固定revision的Qwen3-0.6B模型卡](https://huggingface.co/Qwen/Qwen3-0.6B/blob/c1899de289a04d12100db370d81485cdf75e47ca/README.md)

上面资料支持接口和机制说明；本课实验结果来自实际运行的程序。模拟文本、人工sleep、手动时钟与真实模型结果明确区分。